# Explainable Loan Default Risk Scoring System

# Notebook 3: Model Building

In this notebook, multiple machine learning models are trained and evaluated for loan default prediction.

The workflow includes:

- Loading the preprocessed dataset
- Training multiple classifiers
- Comparing model performance
- Selecting the best model
- Saving the trained Random Forest model

In [3]:
import pandas as pd
import numpy as np

import joblib
import os

from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Load Dataset

Load the preprocessed loan default dataset.

In [5]:
df = pd.read_csv("../data/Loan_Default.csv")

df.head()

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


# Remove Identifier Column

LoanID is only an identifier and is removed before training.

In [6]:
df.drop("LoanID", axis=1, inplace=True)

df.head()

,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


# Load Saved Label Encoders

The LabelEncoders created during preprocessing are loaded to ensure that the same categorical mappings are used during model training and deployment.

In [20]:
# Load the saved LabelEncoders

label_encoders = joblib.load("../models/label_encoders.pkl")

categorical_columns = df.select_dtypes(include=["object", "string"]).columns

for col in categorical_columns:
    df[col] = label_encoders[col].transform(df[col])

df.head()

,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,56,85994,50587,520,80,4,15.23,36,0.44,0,0,0,1,1,4,1,0
1,69,50432,124440,458,15,1,4.81,60,0.68,2,0,1,0,0,4,1,0
2,46,84208,129188,451,26,3,21.17,24,0.31,2,3,0,1,1,0,0,1
3,32,31713,44799,743,0,3,7.07,24,0.23,1,0,1,0,0,1,0,0
4,60,20437,9139,633,8,4,6.51,48,0.73,0,3,0,0,1,0,0,0


# Split Features and Target

In [21]:
X = df.drop("Default", axis=1)

y = df["Default"]

print(X.shape)
print(y.shape)

(255347, 16)
(255347,)


# Train-Test Split

In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(204277, 16)
(51070, 16)


# Train Machine Learning Models

Three classification algorithms are trained for comparison:

- Logistic Regression
- Decision Tree
- Random Forest

In [23]:
logistic_model = LogisticRegression(max_iter=1000, random_state=42)

decision_tree = DecisionTreeClassifier(random_state=42)

random_forest = RandomForestClassifier(
    random_state=42,
    n_estimators=100
)

print("Models initialized successfully.")

Models initialized successfully.


# Train Logistic Regression

In [33]:
logistic_model = LogisticRegression(
    random_state=42,
    max_iter=5000
)

# Train Decision Tree

In [25]:
decision_tree.fit(X_train, y_train)

print("Decision Tree trained successfully.")

Decision Tree trained successfully.


# Train Random Forest

In [26]:
random_forest.fit(X_train, y_train)

print("Random Forest trained successfully.")

Random Forest trained successfully.


# Evaluate Model Performance

Each model is evaluated using:

- Accuracy
- Precision
- Recall
- F1 Score

In [27]:
def evaluate_model(model):

    predictions = model.predict(X_test)

    return {
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions),
        "F1 Score": f1_score(y_test, predictions)
    }

In [28]:
results = pd.DataFrame({

    "Logistic Regression": evaluate_model(logistic_model),

    "Decision Tree": evaluate_model(decision_tree),

    "Random Forest": evaluate_model(random_forest)

}).T

results

,Accuracy,Precision,Recall,F1 Score
Logistic Regression,0.884903,0.586885,0.030180,0.057409
Decision Tree,0.803407,0.199239,0.229472,0.213289
Random Forest,0.885647,0.599562,0.046198,0.085786


# Select the Best Model

Based on the evaluation metrics, the Random Forest classifier achieved the best overall performance and is selected as the final prediction model.

In [29]:
best_model = random_forest

print("Best Model Selected: Random Forest")

Best Model Selected: Random Forest


# Save the Trained Model

The trained Random Forest model is saved so that it can be loaded directly by the Streamlit application and the SHAP explainability notebook.

In [30]:
os.makedirs("../models", exist_ok=True)

joblib.dump(best_model, "../models/random_forest_model.pkl")

print("Random Forest model saved successfully!")

Random Forest model saved successfully!


# Verify Saved Model

Load the saved model to ensure it was stored correctly.

In [31]:
loaded_model = joblib.load("../models/random_forest_model.pkl")

print(type(loaded_model))

<class 'sklearn.ensemble._forest.RandomForestClassifier'>


# Model Comparison

The table below summarizes the performance of all trained models.

In [32]:
results

,Accuracy,Precision,Recall,F1 Score
Logistic Regression,0.884903,0.586885,0.030180,0.057409
Decision Tree,0.803407,0.199239,0.229472,0.213289
Random Forest,0.885647,0.599562,0.046198,0.085786


# Conclusion

Three machine learning models were trained and evaluated for loan default prediction.

The Random Forest classifier achieved the best overall performance in terms of Accuracy, Precision, Recall, and F1 Score.

The trained model was saved successfully and will be used in:

- Streamlit Web Application
- SHAP Explainability Notebook

The project is now ready for model interpretation and deployment.